In [1]:
import os
import gc
import csv
import numpy as np
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from tensorflow.keras.preprocessing import image_dataset_from_directory

# ==============================================================================
# 1. SETUP PATHS & CONFIGURATION
# ==============================================================================
SIPAKMED_VAL_PATH = "/kaggle/input/datasets/sazedkarim/pap-png/png"
# Updated to your specific model directory
MODELS_DIR = "/kaggle/input/models/sazedkarim/sipakmed-pretrained-models-7-oct-2026/tensorflow2/default/1" 
OUTPUT_DIR = "/kaggle/working/model_evaluations"

os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_FILE_PATH = os.path.join(OUTPUT_DIR, "all_models_ood_results.csv")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# ==============================================================================
# 2. OOD DETECTOR CLASS 
# ==============================================================================
class DissonantChordSearch:
    def __init__(self, num_features, top_k):
        self.num_features = num_features
        self.top_k = float(top_k)
        self.co_matrix = tf.Variable(tf.zeros((num_features, num_features), dtype=tf.float32))

    def update_and_finalize(self, activations):
        _, top_indices = tf.math.top_k(activations, k=int(self.top_k))
        mask = tf.one_hot(top_indices, depth=self.num_features)
        mask = tf.reduce_sum(mask, axis=1)
        co_occurrences = tf.matmul(mask, mask, transpose_a=True)
        
        matrix = tf.linalg.set_diag(co_occurrences, tf.zeros(self.num_features))
        row_sums = tf.reduce_sum(matrix, axis=1, keepdims=True)
        self.co_matrix.assign(matrix / (row_sums + 1e-8))

    def score(self, activations):
        batch_size = tf.shape(activations)[0]
        _, top_indices = tf.math.top_k(activations, k=int(self.top_k))
        scores = []
        
        denominator = tf.maximum(self.top_k * (self.top_k - 1.0), 1e-8)
        
        for i in range(batch_size):
            idx = top_indices[i]
            sub_rows = tf.gather(self.co_matrix, idx)
            sub_matrix = tf.gather(sub_rows, idx, axis=1)
            total_prob = tf.reduce_sum(sub_matrix)
            scores.append(total_prob / denominator)
            
        return tf.stack(scores)

# ==============================================================================
# 3. INITIALIZE CSV WRITER
# ==============================================================================
with open(CSV_FILE_PATH, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["Model", "Top_K", "AUROC", "ID_Accuracy", "OOD_Rejection_Rate", "Threshold_10th_Percentile"])

# ==============================================================================
# 4. DATASET LOADING (SIPaKMeD + CIFAR-10)
# ==============================================================================
print("Loading SIPaKMeD ID dataset...")
# The SIPaKMeD dataset consists of 4049 images of isolated cells that have been manually cropped from 966 cluster cell images of Pap smear slides.
val_ds = image_dataset_from_directory(
    SIPAKMED_VAL_PATH, validation_split=0.2, subset="validation", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode=None
).prefetch(tf.data.AUTOTUNE)

print("Loading CIFAR-10 OOD dataset...")
(x_ood, _), _ = tf.keras.datasets.cifar10.load_data()

x_ood = x_ood[:2000] 

ood_dataset = tf.data.Dataset.from_tensor_slices(x_ood)
ood_dataset = ood_dataset.map(lambda x: tf.image.resize(x, IMAGE_SIZE))
ood_dataset = ood_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ==============================================================================
# 5. DYNAMIC MODEL DISCOVERY & MASTER LOOP
# ==============================================================================
# Scan the directory and filter for .keras files
model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.keras')]
print(f"Found {len(model_files)} models in directory.")

for model_file in model_files:
    model_path = os.path.join(MODELS_DIR, model_file)
    model_name = model_file.replace('_final.keras', '')
        
    print(f"\n{'='*50}\n🚀 Processing Model: {model_name} (File: {model_file})\n{'='*50}")
    
    # Initialize tracking variables to None so the finally block never crashes
    full_model = None
    feature_extractor = None
    id_features = None
    ood_features = None
    
    try:
        full_model = tf.keras.models.load_model(model_path)
        
        # Safely extract the penultimate output tensor
        penultimate_tensor = full_model.layers[-2].output
        feature_extractor = tf.keras.Model(inputs=full_model.input, outputs=penultimate_tensor)
        
        # Get the dimension directly from the tensor shape rather than the layer attribute
        feature_dim = penultimate_tensor.shape[-1]
        print(f"Feature Dimension: {feature_dim}")
        
        print("Extracting ID and OOD features...")
        id_features = tf.concat([feature_extractor(batch, training=False) for batch in val_ds], axis=0)
        ood_features = tf.concat([feature_extractor(batch, training=False) for batch in ood_dataset], axis=0)
        
        best_auroc = 0.0
        best_k = 2
        best_matrix = None
        
        with open(CSV_FILE_PATH, mode='a', newline='') as file:
            writer = csv.writer(file)
            
            for k in range(2, 51):
                detector = DissonantChordSearch(num_features=feature_dim, top_k=k)
                detector.update_and_finalize(id_features)
                
                id_scores = detector.score(id_features).numpy()
                ood_scores = detector.score(ood_features).numpy()
                
                y_true = [1] * len(id_scores) + [0] * len(ood_scores)
                y_scores = list(id_scores) + list(ood_scores)
                
                auroc = roc_auc_score(y_true, y_scores)
                threshold = np.percentile(id_scores, 10) 
                id_acc = np.mean(id_scores >= threshold) * 100
                ood_rej = np.mean(ood_scores < threshold) * 100
                
                writer.writerow([model_name, k, f"{auroc:.4f}", f"{id_acc:.2f}", f"{ood_rej:.2f}", f"{threshold:.6f}"])
                
                if auroc > best_auroc:
                    best_auroc = auroc
                    best_k = k
                    best_matrix = detector.co_matrix.numpy()
                    
        print(f"✅ {model_name} Complete! Best Top K: {best_k} (AUROC: {best_auroc:.4f})")
        
        matrix_save_path = os.path.join(OUTPUT_DIR, f"{model_name}_best_matrix_k{best_k}.npy")
        np.save(matrix_save_path, best_matrix)
        
    except Exception as e:
        print(f"❌ Error processing {model_name}: {e}")
        
    finally:
        # Safely clear memory only if the variables were successfully created
        tf.keras.backend.clear_session()
        if full_model is not None:
            del full_model
        if feature_extractor is not None:
            del feature_extractor
        if id_features is not None:
            del id_features
        if ood_features is not None:
            del ood_features
        gc.collect()

print("\n🎉 ALL MODELS PROCESSED. Check the output directory for your CSV.")
print("Reference file noted: image_393921.png")

Loading SIPaKMeD ID dataset...
Found 4049 files.
Using 809 files for validation.


I0000 00:00:1791403017.912924      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791403017.916667      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading CIFAR-10 OOD dataset...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 401s 2us/step
Found 26 models in directory.

🚀 Processing Model: efficientnetb3 (File: efficientnetb3_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
✅ efficientnetb3 Complete! Best Top K: 50 (AUROC: 0.9914)

🚀 Processing Model: vgg16 (File: vgg16_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
✅ vgg16 Complete! Best Top K: 7 (AUROC: 0.9366)

🚀 Processing Model: convnextsmall (File: convnextsmall_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...


I0000 00:00:1791403809.432272      23 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2026-10-07 20:10:14.423657: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 20:10:14.567777: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 20:10:14.702202: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 20:12:14.869341: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 20:12:15.014423: E external/local_xla/xla/stream_

✅ convnextsmall Complete! Best Top K: 14 (AUROC: 0.9989)

🚀 Processing Model: efficientnetv2l (File: efficientnetv2l_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
❌ Error processing efficientnetv2l: Exception encountered when calling Activation.call().

{{function_node __wrapped__Sigmoid_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:Sigmoid] name: 

Arguments received by Activation.call():
  • inputs=tf.Tensor(shape=(32, 7, 7, 2304), dtype=float32)

🚀 Processing Model: densenet121 (File: densenet121_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
✅ densenet121 Complete! Best Top K: 36 (AUROC: 0.9915)

🚀 Processing Model: resnet101v2 (File: resnet101v2_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
✅ resnet101v2 Complete! Best Top K: 2 (AUROC: 0.9384)

🚀 Processing Model: resnet50 (File: resnet50_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
✅ resnet50 Compl

2026-10-07 21:28:01.228963: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 21:28:01.374454: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 21:29:59.123779: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 21:29:59.272139: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-07 21:35:16.663192: E external/local_xla/xla/stream_

✅ convnextbase Complete! Best Top K: 48 (AUROC: 0.9982)

🚀 Processing Model: resnet50v2 (File: resnet50v2_final.keras)
Feature Dimension: 256
Extracting ID and OOD features...
✅ resnet50v2 Complete! Best Top K: 7 (AUROC: 0.9832)

🎉 ALL MODELS PROCESSED. Check the output directory for your CSV.
Reference file noted: image_393921.png
